# Задача 2. Permutations

Дан массив `nums`. Нужно вернуть все возможные перестановки его элементов
и показать стек рекурсивных вызовов.

Вход: `[1, 2, 3]`

Выход: `[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]`

## Объяснение решения

Заполняем перестановку слева направо. `path` хранит выбранные числа,
а `used` показывает, какие позиции исходного массива уже заняты.

Выбираем свободный элемент, добавляем его в `path` и рекурсивно
заполняем оставшиеся места. Когда выбраны все элементы, сохраняем копию.

После возврата удаляем последний элемент из `path` и снимаем отметку
`used`. Теперь можно попробовать следующий вариант. Это поиск с возвратом.

Чтобы одинаковые числа не создавали повторные ответы, сортируем копию
входа и перебираем одинаковые элементы в порядке их индексов.
Сам входной массив не меняется.

Перебор всех свободных вариантов даёт все перестановки.
Отмена выбора позволяет соседним ветвям работать независимо.

## Декоратор из первой задачи

Используем тот же `tracer`. Его код повторён здесь, чтобы ноутбук
можно было запустить отдельно.

In [1]:
from contextvars import ContextVar
from functools import wraps
import sys


# общая глубина для всех функций с декоратором
_depth = ContextVar("tracer_depth", default=0)


def _print_call(output, depth, name, args, kwargs):
    """Строки существуют только во время печати, а не до конца рекурсии."""
    # собираем позиционные и именованные аргументы для вывода
    arguments = [repr(value) for value in args]
    arguments.extend(f"{key}={value!r}" for key, value in kwargs.items())
    print(f"{'  ' * depth}-> {name}({', '.join(arguments)})", file=output)


def _print_return(output, depth, name, result):
    print(f"{'  ' * depth}<- {name} = {result!r}", file=output)


def _print_exception(output, depth, name, error):
    print(
        f"{'  ' * depth}!! {name} raised {type(error).__name__}: {error}",
        file=output,
    )


def tracer(func=None, *, enabled=True, stream=None):
    """Показать входы, возвраты и исключения с отступом по глубине вызова.

    Поддерживаются обе формы: ``@tracer`` и ``@tracer(stream=...)``.
    ``enabled=False`` возвращает исходную функцию без трассировки.
    При ``stream=None`` текущий ``sys.stdout`` выбирается при каждом вызове.
    Декоратор предназначен для обычных синхронных функций.
    """

    def decorate(function):
        # если трассировка выключена, возвращаем функцию без обёртки
        if not enabled:
            return function

        # сохраняем имя и описание исходной функции
        @wraps(function)
        def wrapper(*args, **kwargs):
            output = sys.stdout if stream is None else stream
            depth = _depth.get()

            # показываем вход и увеличиваем глубину для вложенных вызовов
            _print_call(output, depth, function.__name__, args, kwargs)
            token = _depth.set(depth + 1)
            try:
                result = function(*args, **kwargs)
            except BaseException as error:
                # показываем ошибку и передаём её дальше
                _print_exception(output, depth, function.__name__, error)
                raise
            else:
                # показываем результат и возвращаем его без изменения
                _print_return(output, depth, function.__name__, result)
                return result
            finally:
                # восстанавливаем глубину даже после ошибки
                _depth.reset(token)

        return wrapper

    # поддерживаем запись и @tracer, и @tracer(...)
    if func is None:
        return decorate
    return decorate(func)

In [2]:
def permutations(nums, *, trace=False, stream=None):
    """Вернуть все уникальные перестановки в лексикографическом порядке.

    Входная последовательность не изменяется. У пустой последовательности
    одна перестановка — пустая последовательность. При trace=True входы
    и возвраты рекурсивной функции печатаются декоратором из первой задачи.
    Число при возврате — количество завершённых перестановок в этой ветви.
    """
    values = sorted(nums)  # Сортируем копию, исходный массив не меняем.
    used = [False] * len(values)  # Какие позиции уже заняты в текущем пути.
    result = []

    @tracer(enabled=trace, stream=stream)
    def backtrack(path):
        if len(path) == len(values):
            # Все места заполнены. Сохраняем копию готового ответа.
            result.append(path.copy())
            return 1

        completed = 0
        for i, value in enumerate(values):
            if used[i]:
                continue

            # Пропускаем одинаковый выбор, чтобы не повторять перестановки.
            if i > 0 and value == values[i - 1] and not used[i - 1]:
                continue

            # Выбираем число и достраиваем оставшуюся часть.
            used[i] = True
            path.append(value)

            completed += backtrack(path)

            # Отменяем выбор и пробуем следующее число.
            path.pop()
            used[i] = False

        return completed

    backtrack([])
    return result

In [3]:
print(permutations([1, 2, 3]))
print(permutations([0, 1]))
print(permutations([1]))
print(permutations([]))
print(permutations([1, 1, 2]))

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
[[0, 1], [1, 0]]
[[1]]
[[]]
[[1, 1, 2], [1, 2, 1], [2, 1, 1]]


## Визуализация работы алгоритма

Для `[1, 2, 3]` получается дерево выбора:

```text
[]
├── [1]
│   ├── [1, 2] → [1, 2, 3]
│   └── [1, 3] → [1, 3, 2]
├── [2]
│   ├── [2, 1] → [2, 1, 3]
│   └── [2, 3] → [2, 3, 1]
└── [3]
    ├── [3, 1] → [3, 1, 2]
    └── [3, 2] → [3, 2, 1]
```

Сначала выбираем 1, затем 2, затем 3. Сохраняем `[1, 2, 3]`.
Возвращаемся к `[1]`, выбираем 3 вместо 2 и получаем `[1, 3, 2]`.
После всех вариантов с первой единицей переходим к первой двойке.

В трассировке возврат `1` означает одну готовую перестановку в ветви,
возврат `2` из `[1]` — два ответа с первой единицей.
Последний возврат `6` — общее число ответов.

In [4]:
result = permutations([1, 2, 3], trace=True)

print('Результат:', result)
print('Количество перестановок:', len(result))

-> backtrack([])
  -> backtrack([1])
    -> backtrack([1, 2])
      -> backtrack([1, 2, 3])
      <- backtrack = 1
    <- backtrack = 1
    -> backtrack([1, 3])
      -> backtrack([1, 3, 2])
      <- backtrack = 1
    <- backtrack = 1
  <- backtrack = 2
  -> backtrack([2])
    -> backtrack([2, 1])
      -> backtrack([2, 1, 3])
      <- backtrack = 1
    <- backtrack = 1
    -> backtrack([2, 3])
      -> backtrack([2, 3, 1])
      <- backtrack = 1
    <- backtrack = 1
  <- backtrack = 2
  -> backtrack([3])
    -> backtrack([3, 1])
      -> backtrack([3, 1, 2])
      <- backtrack = 1
    <- backtrack = 1
    -> backtrack([3, 2])
      -> backtrack([3, 2, 1])
      <- backtrack = 1
    <- backtrack = 1
  <- backtrack = 2
<- backtrack = 6
Результат: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
Количество перестановок: 6


## Оценка сложности

Пусть `n` — количество элементов. Оценки ниже не включают печать стека.

**Время в худшем случае: `O(n · n!)`**

Для `n` разных чисел существует `n!` перестановок.
Каждую копируем в ответ за `O(n)`. Перебор свободных позиций укладывается
в ту же оценку. Сортировка `O(n log n)` итоговую оценку не меняет.

**Дополнительная память без результата: `O(n)`**

Храним отсортированную копию, `used`, текущий путь и стек глубины `n`.

**Память результата в худшем случае: `O(n · n!)`**

Повторяющиеся числа уменьшают количество различных ответов.
Для пустого массива возвращаем `[[]]`: одну пустую перестановку.
Трассировка дополнительно требует времени на форматирование и вывод;
при сохранении всего текста нужна память для этого текста.

## Тесты

Проверяем примеры из условия, пустой массив, повторы, отрицательные числа,
неизменность входа и вывод стека. На небольших массивах сравниваем
результат с `itertools.permutations`.

In [5]:
import io
import itertools
import unittest
from contextlib import redirect_stdout



class TestPermutations(unittest.TestCase):
    def test_three_elements_from_statement(self):
        self.assertEqual(
            permutations([1, 2, 3]),
            [
                [1, 2, 3],
                [1, 3, 2],
                [2, 1, 3],
                [2, 3, 1],
                [3, 1, 2],
                [3, 2, 1],
            ],
        )

    def test_two_elements_from_statement(self):
        self.assertEqual(permutations([0, 1]), [[0, 1], [1, 0]])

    def test_one_element_from_statement(self):
        self.assertEqual(permutations([1]), [[1]])

    def test_empty_input_has_one_empty_permutation(self):
        self.assertEqual(permutations([]), [[]])

    def test_repeated_values_are_unique(self):
        self.assertEqual(
            permutations([1, 1, 2]), [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
        )

    def test_all_values_equal(self):
        self.assertEqual(permutations([7, 7, 7]), [[7, 7, 7]])

    def test_negative_unsorted_input_is_returned_in_lexicographic_order(self):
        self.assertEqual(
            permutations([2, -1, 0]),
            [
                [-1, 0, 2],
                [-1, 2, 0],
                [0, -1, 2],
                [0, 2, -1],
                [2, -1, 0],
                [2, 0, -1],
            ],
        )

    def test_input_is_not_changed(self):
        nums = [3, 1, 2, 1]
        before = nums.copy()
        permutations(nums)
        self.assertEqual(nums, before)

    def test_result_lists_do_not_share_state(self):
        nums = [1, 2]
        result = permutations(nums)
        result[0][0] = 99
        self.assertEqual(result[1], [2, 1])
        self.assertEqual(nums, [1, 2])

    def test_repeated_calls_do_not_accumulate_results(self):
        first = permutations([1, 2])
        self.assertEqual(permutations([]), [[]])
        self.assertEqual(permutations([2, 1]), first)

    def test_matches_standard_library_oracle(self):
        cases = [[], [0], [2, 1], [1, 1, 2], [-2, 0, -2, 1], [3, 2, 1, 0]]
        for nums in cases:
            with self.subTest(nums=nums):
                expected = [list(item) for item in sorted(set(itertools.permutations(nums)))]
                self.assertEqual(permutations(nums), expected)

    def test_tracing_is_disabled_by_default(self):
        captured = io.StringIO()
        with redirect_stdout(captured):
            result = permutations([1, 2])
        self.assertEqual(result, [[1, 2], [2, 1]])
        self.assertEqual(captured.getvalue(), "")

    def test_trace_shows_calls_depth_and_returned_counts(self):
        stream = io.StringIO()
        result = permutations([1, 2], trace=True, stream=stream)
        self.assertEqual(result, [[1, 2], [2, 1]])
        self.assertEqual(
            stream.getvalue().splitlines(),
            [
                "-> backtrack([])",
                "  -> backtrack([1])",
                "    -> backtrack([1, 2])",
                "    <- backtrack = 1",
                "  <- backtrack = 1",
                "  -> backtrack([2])",
                "    -> backtrack([2, 1])",
                "    <- backtrack = 1",
                "  <- backtrack = 1",
                "<- backtrack = 2",
            ],
        )

    def test_trace_for_three_elements_reports_six_completed_permutations(self):
        stream = io.StringIO()
        result = permutations([1, 2, 3], trace=True, stream=stream)
        lines = stream.getvalue().splitlines()
        self.assertEqual(len(result), 6)
        self.assertEqual(lines[-1], "<- backtrack = 6")
        self.assertEqual(sum(line.lstrip().startswith("->") for line in lines), 16)
        self.assertEqual(sum(line.lstrip().startswith("<-") for line in lines), 16)

    def test_trace_of_empty_input(self):
        stream = io.StringIO()
        self.assertEqual(permutations([], trace=True, stream=stream), [[]])
        self.assertEqual(
            stream.getvalue().splitlines(), ["-> backtrack([])", "<- backtrack = 1"]
        )


import sys

suite = unittest.TestLoader().loadTestsFromTestCase(TestPermutations)
runner = unittest.TextTestRunner(stream=sys.stdout, verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful()

test_all_values_equal (__main__.TestPermutations.test_all_values_equal) ... 

ok


test_empty_input_has_one_empty_permutation (__main__.TestPermutations.test_empty_input_has_one_empty_permutation) ... 

ok


test_input_is_not_changed (__main__.TestPermutations.test_input_is_not_changed) ... 

ok


... (21 outputs left out)

ok


test_two_elements_from_statement (__main__.TestPermutations.test_two_elements_from_statement) ... 

ok


----------------------------------------------------------------------
Ran 15 tests in 0.004s

OK
